# Deterministic Response Engineering for Production APIs

## What you will build

You will build an invoice parsing service for an accounts payable team. Suppliers send invoices as
text, a model reads each one, and your code posts what it read to the ledger the bank pays from. The
ledger needs the same exact fields every time, so the service forces every reply from the model
into one fixed shape instead of asking for that shape in words.

A fixed shape stops only some of the mistakes. The diagram shows the three this course stops: a
reply the parser cannot read at all, a Swiss invoice posted in euros because euros were an allowed
answer, and a receipt for an invoice that was already paid, posted as a new invoice so the bank
would pay it twice.

![What you will build](images/invoice-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it replays
responses recorded from real runs, so you can follow the whole course for free, and with a key it
calls the model live. The cell also reads the provider's published facts to confirm that this model
accepts `tool_choice`, because the whole course depends on that one setting.

In [1]:
import copy
import json
import re
from decimal import Decimal, InvalidOperation
from types import SimpleNamespace

from vault import get_client, load_env, model_for, provider_truth

load_env()
client = get_client("08-deterministic-outputs/01-build-an-invoice-extractor")
MODEL = model_for("default")

supported = provider_truth()["models"][MODEL]["supported_parameters"]
print(f"Client ready. Every request in this notebook uses {MODEL}.")
for setting in ("tools", "tool_choice", "structured_outputs"):
    print(f"  {setting:18} supported: {setting in supported}")

Client ready. Every request in this notebook uses google/gemini-2.5-flash-lite.
  tools              supported: True
  tool_choice        supported: True
  structured_outputs supported: True


## Step 1: Create the inbox and the payables ledger

The service needs real documents to read, so we start with five that arrive in one morning's inbox.
Two are ordinary invoices, one is in Swiss francs, one is a remittance advice saying an invoice was
already paid, and one is a scan too damaged to read.

In [2]:
INVOICES = {
    "northwind": ("Northwind Office Supply\nInvoice NW-20931, 2 March 2026\n"
                  "3 x toner cartridge @ 89.00 = 267.00\n1 x paper, one case = 42.50\n"
                  "Subtotal 309.50\nSales tax 24.76\nTotal due USD 334.26"),
    "brandt": ("Brandt Logistik GmbH, Hamburg\nRechnung / Invoice RE-7731\n"
               "Fracht Hamburg - Rotterdam 1.240,50 EUR\nHandling 85,00 EUR\n"
               "Netto 1.325,50 EUR\nMwSt 19% 251,85 EUR\nGesamtbetrag 1.577,35 EUR"),
    "alpenlicht": ("Alpenlicht AG, Zurich\nInvoice 2026-114\nLighting installation, lobby\n"
                   "Total CHF 4'800.00 including VAT"),
    "remittance": ("Remittance advice from Contoso Ltd\nWe have paid your invoice NW-20931 "
                   "for USD 334.26 on 12 March 2026. No further action is needed. Thank you."),
    "unreadable": "INV0lCE ####  ;; T0TAL ..,.. %%% 0x3F 0x3F [scan truncated] ....",
}

print(f"{len(INVOICES)} documents in the inbox: {list(INVOICES)}")

5 documents in the inbox: ['northwind', 'brandt', 'alpenlicht', 'remittance', 'unreadable']


The ledger is the list the bank pays from. The payments team can pay in the three currencies in
`PAYABLE_CURRENCIES`, and `post_invoice` records every invoice it is given, whatever that invoice
says.

In [3]:
PAYABLE_CURRENCIES = ("USD", "EUR", "GBP")
PAYABLES = []   # every invoice the bank will pay


def post_invoice(fields):
    """Add one invoice to the ledger. The bank pays whatever is in this list."""
    row = {key: fields[key] for key in ("vendor", "invoice_number", "currency", "total_cents")}
    PAYABLES.append(row)
    return row


print(f"The ledger pays in {PAYABLE_CURRENCIES} and holds {len(PAYABLES)} invoices so far.")

The ledger pays in ('USD', 'EUR', 'GBP') and holds 0 invoices so far.


## Step 2: Ask for JSON in words and parse the reply

The first thing most people try is to describe the JSON they want in the system prompt and parse
whatever comes back. We try that on all five documents and count how many replies `json.loads` can
read.

In [4]:
SYSTEM_PROMPT = "You parse supplier invoices for an accounts payable team."
JSON_IN_WORDS = (" Reply with JSON only, no prose and no markdown, in this shape: "
                 '{"vendor": str, "invoice_number": str, "currency": str, "total": number}')


def ask_for_json_in_words(document):
    """Describe the JSON in the prompt and return the reply text exactly as it came back."""
    response = client.chat.completions.create(
        model=MODEL, max_tokens=300,
        messages=[{"role": "system", "content": SYSTEM_PROMPT + JSON_IN_WORDS},
                  {"role": "user", "content": document}])
    return response.choices[0].message.content


print("ask_for_json_in_words sends the shape as an instruction in the system prompt")

ask_for_json_in_words sends the shape as an instruction in the system prompt


The parser is the ordinary one, and a reply it cannot read is counted instead of crashing the loop.

In [5]:
parsed = 0
for name, document in INVOICES.items():
    reply = ask_for_json_in_words(document)
    try:
        json.loads(reply)
        parsed += 1
        print(f"{name:11} parsed")
    except json.JSONDecodeError as error:
        print(f"{name:11} failed: {error.msg}, the reply starts {reply[:10]!r}")

print(f"\n{parsed} of {len(INVOICES)} replies parsed")

northwind   failed: Expecting value, the reply starts '```json\n{\n'


brandt      failed: Expecting value, the reply starts '```json\n{"'


alpenlicht  failed: Expecting value, the reply starts '```json\n{"'


remittance  failed: Expecting value, the reply starts '```json\n{"'


unreadable  failed: Expecting value, the reply starts '```json\n{"'

0 of 5 replies parsed


None of the five replies parsed, although the prompt said no markdown in plain words. The model
wrapped every answer in a markdown code fence, so the first character `json.loads` met was a
backtick. An instruction about format travels in the same text as the rest of the prompt, so nothing
in the request makes the model follow it.

## Step 3: Force every reply through the extract_invoice tool

A format asked for in words is only a request, so we move the shape out of the prompt and into the
request itself. We describe an `extract_invoice` tool and set **tool_choice**, the setting that
forces the model to answer through a named tool, so the reply must arrive as a tool call and never
as prose.

In [6]:
FIRST_PASS_TOOL = {"type": "function", "function": {
    "name": "extract_invoice",
    "description": "Record one supplier invoice for accounts payable.",
    "parameters": {"type": "object", "properties": {
        "vendor": {"type": "string"}, "invoice_number": {"type": "string"},
        "currency": {"type": "string"}, "total": {"type": "string"}}}}}


def build_extract_request(document, tool):
    """One place decides how the model may answer: through extract_invoice, always."""
    return {"model": MODEL, "max_tokens": 300, "tools": [tool],
            "tool_choice": {"type": "function", "function": {"name": "extract_invoice"}},
            "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": document}]}


print(json.dumps(build_extract_request("...", FIRST_PASS_TOOL)["tool_choice"]))

{"type": "function", "function": {"name": "extract_invoice"}}


The parameters of the tool are its **schema**, the written shape of the allowed arguments, and this
first version is the one most people write first: every field is a string and nothing is required.
`extract_invoice_fields` sends the forced request and parses the arguments, which still arrive as a
JSON string.

In [7]:
def extract_invoice_fields(document, tool):
    """Send the forced request. Return the choice and the first tool call's parsed arguments."""
    choice = client.chat.completions.create(**build_extract_request(document, tool)).choices[0]
    return choice, json.loads(choice.message.tool_calls[0].function.arguments)


first_pass = {}
for name, document in INVOICES.items():
    choice, first_pass[name] = extract_invoice_fields(document, FIRST_PASS_TOOL)
    print(f"{name:11} finish_reason={choice.finish_reason:10} "
          f"tool calls={len(choice.message.tool_calls)}  {first_pass[name]}")

northwind   finish_reason=tool_calls tool calls=1  {'vendor': 'Northwind Office Supply', 'total': '334.26', 'invoice_number': 'NW-20931', 'currency': 'USD'}


brandt      finish_reason=tool_calls tool calls=1  {'total': '1.577,35', 'vendor': 'Brandt Logistik GmbH', 'currency': 'EUR', 'invoice_number': 'RE-7731'}


alpenlicht  finish_reason=tool_calls tool calls=1  {'currency': 'CHF', 'invoice_number': '2026-114', 'vendor': 'Alpenlicht AG', 'total': '4800.00'}


remittance  finish_reason=tool_calls tool calls=2  {'currency': 'USD', 'total': '334.26', 'invoice_number': 'NW-20931', 'vendor': 'Contoso Ltd'}


unreadable  finish_reason=tool_calls tool calls=1  {'vendor': 'UNKNOWN', 'currency': 'USD', 'total': '0x3F 0x3F', 'invoice_number': 'INV0lCE #### 0x3F 0x3F'}


All five replies came back as tool calls, and every one of them parsed. The **finish_reason**, which
is the field on a response that says why the model stopped talking, reads `tool_calls` each time.
The remittance reply held two tool calls instead of one, and that matters later, because
`tool_calls` is a list and this code reads only its first entry.

## Step 4: Check that every reply parses and its types match

A reply that parses is not yet a reply our code can use, so the next check compares it with the
schema it was sent. **Syntactic validity** means exactly this: the JSON parses, every required key
is present, every value has the declared type, and no key appears that the schema did not allow.

![Check that every reply parses and its types match](images/invoice-extractor-step-1.svg)

In [8]:
JSON_TYPES = {"string": str, "integer": int, "null": type(None)}


def find_schema_errors(fields, tool):
    """List every way the parsed arguments break the tool's schema. Empty means valid."""
    schema = tool["function"]["parameters"]
    errors = [f"missing {key}" for key in schema.get("required", []) if key not in fields]
    for key, value in fields.items():
        rule = schema["properties"].get(key)
        if rule is None:
            if schema.get("additionalProperties") is False:
                errors.append(f"unexpected key {key}")
            continue
        allowed = rule["type"] if isinstance(rule["type"], list) else [rule["type"]]
        if not any(type(value) is JSON_TYPES[kind] for kind in allowed):
            errors.append(f"{key} is {type(value).__name__}, the schema says {rule['type']}")
        if "enum" in rule and value not in rule["enum"]:
            errors.append(f"{key} {value!r} is not one of {rule['enum']}")
    return errors


for name, fields in first_pass.items():
    print(f"{name:11} schema errors: {find_schema_errors(fields, FIRST_PASS_TOOL)}")

northwind   schema errors: []
brandt      schema errors: []
alpenlicht  schema errors: []
remittance  schema errors: []
unreadable  schema errors: []


Every reply is valid, because the first-pass schema accepts any string in any field. The ledger
needs the total in cents, so the service now has to convert text it never chose, and
`convert_total_to_cents` is where that conversion happens.

In [9]:
def convert_total_to_cents(total_text):
    """Turn the model's total, copied as text, into integer cents for the ledger."""
    return int(Decimal(total_text) * 100)


for name, fields in first_pass.items():
    try:
        cents = convert_total_to_cents(fields.get("total"))
        print(f"{name:11} {fields.get('currency')} {cents} cents")
    except (InvalidOperation, TypeError) as error:
        print(f"{name:11} cannot convert {fields.get('total')!r}: {type(error).__name__}")

northwind   USD 33426 cents
brandt      cannot convert '1.577,35': InvalidOperation
alpenlicht  CHF 480000 cents
remittance  USD 33426 cents
unreadable  cannot convert '0x3F 0x3F': InvalidOperation


Two of the five totals cannot be converted, and both are perfectly valid strings. The German total
came back as `1.577,35`, copied with its German separators, and the damaged scan came back with a
total of `0x3F 0x3F`. The Swiss total converted cleanly, but into cents of a currency the ledger
cannot pay. A loose schema makes a reply easy to validate and hard to use, because it lets the model
put anything in any field.

## Step 5: Lock the schema down so only usable values fit

Each rule the ledger relies on now moves into the schema, so the model is held to it on every call
instead of our code guessing afterwards. This is **structural lock down**: money as integer cents,
the currency as an `enum` of the three payable codes, every key `required`, `additionalProperties`
set to false so no unlisted key can arrive, and `strict` asking the provider to hold the arguments
to the schema exactly.

![Lock the schema down so only usable values fit](images/invoice-extractor-step-2.svg)

In [10]:
LOCKED_TOOL = {"type": "function", "function": {
    "name": "extract_invoice",
    "description": "Record one supplier invoice for accounts payable.",
    "strict": True,
    "parameters": {"type": "object", "properties": {
        "vendor": {"type": "string"},
        "invoice_number": {"type": "string"},
        "currency": {"type": "string", "enum": list(PAYABLE_CURRENCIES)},
        "subtotal_cents": {"type": "integer"},
        "tax_cents": {"type": "integer"},
        "total_cents": {"type": "integer"}},
        "required": ["vendor", "invoice_number", "currency",
                     "subtotal_cents", "tax_cents", "total_cents"],
        "additionalProperties": False}}}

print(f"every key required: {LOCKED_TOOL['function']['parameters']['required']}")

every key required: ['vendor', 'invoice_number', 'currency', 'subtotal_cents', 'tax_cents', 'total_cents']


The same five documents go through the same forced call. Each reply is checked against the locked
schema, and only a valid one reaches the ledger.

In [11]:
PAYABLES.clear()
locked, posted = {}, {}
for name, document in INVOICES.items():
    choice, locked[name] = extract_invoice_fields(document, LOCKED_TOOL)
    errors = find_schema_errors(locked[name], LOCKED_TOOL)
    posted[name] = None if errors else post_invoice(locked[name])
    print(f"{name:11} valid={not errors!s:5} {locked[name].get('currency')} "
          f"{locked[name].get('total_cents'):>7} cents  {locked[name].get('invoice_number')!r}")

print(f"\n{len(PAYABLES)} of {len(INVOICES)} documents posted to the ledger")

northwind   valid=True  USD   33426 cents  'NW-20931'


brandt      valid=True  EUR  157735 cents  'RE-7731'


alpenlicht  valid=True  EUR  480000 cents  '2026-114'


remittance  valid=True  USD   33426 cents  'NW-20931'


unreadable  valid=True  USD    3905 cents  'INV0lCE ####'

5 of 5 documents posted to the ledger


Every reply is now valid against a schema that allows only values the ledger can use, and the
German total arrived as 157735 cents with no conversion code at all.

## Step 6: Measure how many valid replies are actually correct

Passing the schema proves that a reply has the right shape, and it proves nothing about whether the
values are true. We compare what the ledger now holds with what a person reading the same five
documents would have posted, which `CORRECT_POSTINGS` writes down.

In [12]:
CORRECT_POSTINGS = {        # what a person would post; None means it must not be paid
    "northwind": ("USD", 33426),
    "brandt": ("EUR", 157735),
    "alpenlicht": None,     # Swiss francs, which the payments team cannot pay
    "remittance": None,     # a receipt for NW-20931, which is already paid
    "unreadable": None,     # nothing in the scan can be read
}


def count_correct_outcomes(posted):
    """How many documents the ledger handled the way a person reading them would have."""
    correct = 0
    for name, expected in CORRECT_POSTINGS.items():
        row = posted.get(name)
        correct += (row and (row["currency"], row["total_cents"])) == expected
    return correct


print(f"valid against the locked schema : {len(PAYABLES)} of {len(INVOICES)}")
print(f"handled the way a person would  : {count_correct_outcomes(posted)} of {len(INVOICES)}")

valid against the locked schema : 5 of 5
handled the way a person would  : 2 of 5


The ledger itself shows what the bank would pay this morning.

In [13]:
for row in PAYABLES:
    print(row)

{'vendor': 'Northwind Office Supply', 'invoice_number': 'NW-20931', 'currency': 'USD', 'total_cents': 33426}
{'vendor': 'Brandt Logistik GmbH', 'invoice_number': 'RE-7731', 'currency': 'EUR', 'total_cents': 157735}
{'vendor': 'Alpenlicht AG', 'invoice_number': '2026-114', 'currency': 'EUR', 'total_cents': 480000}
{'vendor': 'Contoso Ltd', 'invoice_number': 'NW-20931', 'currency': 'USD', 'total_cents': 33426}
{'vendor': 'Various', 'invoice_number': 'INV0lCE ####', 'currency': 'USD', 'total_cents': 3905}


Three of the five postings are wrong, and every one of them passed the schema. The Swiss invoice was
posted in euros, because the `enum` offered no franc and the model picked an allowed code instead.
The remittance advice was posted as invoice NW-20931, which the ledger already held from Northwind,
so the bank would pay it twice. The damaged scan became a 3905 cent invoice from a vendor called
Various, and none of those values appear in the scan.

Each of those replies is syntactically valid and still wrong. A schema describes a shape, so it can
reject a value of the wrong kind, but never a value of the right kind that is untrue.

## Step 7: Give the model a way to refuse a document

A forced call with a locked schema leaves the model one legal move, which is to fill in an invoice,
so it fills one in even from a receipt or a damaged scan. We keep `tool_choice` forced to
`extract_invoice` and add a `document_type` field whose values let the model say the text is not an
invoice or cannot be read, with every other field allowed to be null.

![Give the model a way to refuse a document](images/invoice-extractor-step-3.svg)

In [14]:
REFUSING_TOOL = copy.deepcopy(LOCKED_TOOL)
properties = REFUSING_TOOL["function"]["parameters"]["properties"]
for key in ("vendor", "invoice_number", "subtotal_cents", "tax_cents", "total_cents"):
    properties[key]["type"] = [properties[key]["type"], "null"]
properties["currency"] = {"type": ["string", "null"], "enum": [*PAYABLE_CURRENCIES, None],
                          "description": "null when the invoice is in any other currency"}
properties["document_type"] = {
    "type": "string", "enum": ["invoice", "not_an_invoice", "unreadable"],
    "description": "invoice only when the text is a readable supplier invoice"}
REFUSING_TOOL["function"]["parameters"]["required"].append("document_type")
REFUSING_TOOL["function"]["description"] += " Leave a field null unless the text states it."

print(f"document_type can be {properties['document_type']['enum']}")

document_type can be ['invoice', 'not_an_invoice', 'unreadable']


A reply can also carry several tool calls, as the remittance reply did in Step 3. `read_one_answer`
treats identical repeats as one answer and rejects a reply whose calls disagree, and
`extract_invoice_answer` is the forced call again, reading its reply through that function.

In [15]:
class ConflictingAnswersError(Exception):
    """One reply held several tool calls that disagree, so none of them can be trusted."""


def read_one_answer(choice):
    """Parse every tool call. Identical repeats count as one answer; disagreement raises."""
    answers = {json.dumps(json.loads(call.function.arguments), sort_keys=True)
               for call in choice.message.tool_calls}
    if len(answers) > 1:
        raise ConflictingAnswersError(f"{len(answers)} different answers in one reply")
    return json.loads(answers.pop())


def extract_invoice_answer(document, tool):
    """The forced call again, with its reply read through read_one_answer."""
    choice = client.chat.completions.create(**build_extract_request(document, tool)).choices[0]
    return len(choice.message.tool_calls), read_one_answer(choice)


print("a reply now yields exactly one answer, or raises ConflictingAnswersError")

a reply now yields exactly one answer, or raises ConflictingAnswersError


An empty upload is a real edge case in any inbox, so it goes through the locked schema from Step 5
first.

In [16]:
try:
    calls, answer = extract_invoice_answer("", LOCKED_TOOL)
    print(f"empty upload: {calls} tool calls, one answer: {answer}")
except ConflictingAnswersError as error:
    print(f"empty upload rejected: {error}")

empty upload rejected: 4 different answers in one reply


The empty upload came back with four different answers in one reply, from a document that holds
nothing at all. `read_one_answer` rejected the whole reply, so none of the four could reach the
ledger. Next, all six documents, the empty upload included, go through the schema that can refuse.

In [17]:
INBOX = {**INVOICES, "empty": ""}
answers = {}
for name, document in INBOX.items():
    calls, answers[name] = extract_invoice_answer(document, REFUSING_TOOL)
    print(f"{name:11} calls={calls}  {answers[name]['document_type']:15} "
          f"{answers[name]['currency']} {answers[name]['total_cents']}")

northwind   calls=1  invoice         USD 33426


brandt      calls=1  invoice         EUR 157735


alpenlicht  calls=1  invoice         None 480000


remittance  calls=1  not_an_invoice  USD 33426


unreadable  calls=1  unreadable      None None


empty       calls=5  not_an_invoice  None None


The remittance advice came back as `not_an_invoice` and the scan as `unreadable`. The empty upload
came back as `not_an_invoice` in five identical tool calls, which `read_one_answer` counted as one
answer. The Swiss invoice still came back as an invoice, with its currency set to null because no
payable code fits, and a null currency is a value the schema allows.

## Step 8: Check every answer against the document itself

A schema can only say which values are allowed, so it cannot tell a true value from a false one of
the same kind. In Step 5 the model returned EUR for the Swiss invoice, which every schema check
accepts. A **validator** is code that rejects a value that is the right shape but the wrong answer,
and ours checks each value against the text it came from and against the ledger.

![Check every answer against the document itself](images/invoice-extractor-step-4.svg)

The first piece, `find_amounts_in_text`, reads every amount written in the document into cents,
whatever separators the supplier used, so a value the model invented has nothing to match.

In [18]:
AMOUNT = re.compile(r"\d[\d.,']*")


def find_amounts_in_text(document):
    """Every amount written in the document, in cents, whatever separators it uses."""
    amounts = set()
    for written in AMOUNT.findall(document):
        written = written.rstrip(".,'")
        digits = re.sub(r"\D", "", written)
        has_cents = len(written) > 3 and written[-3] in ".,"
        amounts.add(int(digits) if has_cents else int(digits) * 100)
    return amounts


print(sorted(find_amounts_in_text(INVOICES["brandt"])))

[1900, 8500, 25185, 124050, 132550, 157735, 773100]


The list also holds numbers that are not money, such as the 19 of the tax rate, which does no harm
because the check only asks whether a value appears. `find_answer_errors` is the validator, and each
of its rules names a mistake this notebook has already seen happen.

In [19]:
def find_answer_errors(fields, document):
    """List every value the document or the ledger contradicts. Empty means post it."""
    errors = []
    amounts = find_amounts_in_text(document)
    for key in ("subtotal_cents", "tax_cents", "total_cents"):
        if fields[key] is not None and fields[key] not in amounts:
            errors.append(f"{key} {fields[key]} is not in the document")
    if fields["total_cents"] is None:
        errors.append("no total")
    if fields["currency"] is None:
        errors.append("no payable currency")
    elif fields["currency"] not in re.findall(r"\b[A-Z]{3}\b", document):
        errors.append(f"currency {fields['currency']} is not in the document")
    if not fields["invoice_number"] or fields["invoice_number"] not in document:
        errors.append(f"invoice number {fields['invoice_number']!r} is not in the document")
    if any(row["invoice_number"] == fields["invoice_number"] for row in PAYABLES):
        errors.append(f"{fields['invoice_number']} is already in the ledger")
    return errors


PAYABLES.clear()
for label, fields in (("Step 5 reply", locked["alpenlicht"]), ("Step 7 reply", answers["alpenlicht"])):
    print(f"{label}: {find_answer_errors(fields, INVOICES['alpenlicht'])}")

Step 5 reply: ['subtotal_cents 436364 is not in the document', 'tax_cents 43636 is not in the document', 'currency EUR is not in the document']
Step 7 reply: ['no payable currency']


The Step 5 reply breaks three rules, because its subtotal of 436364 cents and its tax of 43636 cents
appear nowhere in the invoice, and EUR is not written on it either. The Step 7 reply breaks one rule,
because it has no payable currency, so the Swiss invoice goes to a person either way.

`route_answer` puts the validator between the model and the ledger. A refusal or a failed check goes
to a person along with its reasons, and only a clean answer is posted.

In [20]:
NEEDS_REVIEW = []   # documents a person must look at, with the reasons


def route_answer(name, fields, document):
    """Post a clean invoice. Send a refusal or a failed check to a person, with reasons."""
    if fields["document_type"] != "invoice":
        reasons = [f"the model refused it as {fields['document_type']}"]
    else:
        reasons = find_answer_errors(fields, document)
    if reasons:
        NEEDS_REVIEW.append({"document": name, "reasons": reasons})
        return None
    return post_invoice(fields)


print("route_answer posts a clean answer and sends everything else to NEEDS_REVIEW")

route_answer posts a clean answer and sends everything else to NEEDS_REVIEW


The six answers from Step 7 now go through `route_answer`, and the same measure from Step 6 counts
how many were handled the way a person would have handled them.

In [21]:
PAYABLES.clear()
NEEDS_REVIEW.clear()
CORRECT_POSTINGS["empty"] = None
routed = {name: route_answer(name, answers[name], INBOX[name]) for name in INBOX}

for name, row in routed.items():
    print(f"{name:11} posted {row}" if row else f"{name:11} sent to review")
for item in NEEDS_REVIEW:
    print(f"  review {item['document']:11} {'; '.join(item['reasons'])}")
print(f"\nposted {len(PAYABLES)} of {len(INBOX)}, handled the way a person would: "
      f"{count_correct_outcomes(routed)} of {len(INBOX)}")

northwind   posted {'vendor': 'Northwind Office Supply', 'invoice_number': 'NW-20931', 'currency': 'USD', 'total_cents': 33426}
brandt      posted {'vendor': 'Brandt Logistik GmbH', 'invoice_number': 'RE-7731', 'currency': 'EUR', 'total_cents': 157735}
alpenlicht  sent to review
remittance  sent to review
unreadable  sent to review
empty       sent to review
  review alpenlicht  no payable currency
  review remittance  the model refused it as not_an_invoice
  review unreadable  the model refused it as unreadable
  review empty       the model refused it as not_an_invoice

posted 2 of 6, handled the way a person would: 6 of 6


Two invoices were posted and four documents went to a person, each with the reason it was stopped.
The model refused three of them itself, and the validator stopped the Swiss invoice because it has
no payable currency. Both rows below were printed by the cells above.

| What stands between the model and the ledger | Posted | Handled the way a person would |
|---|---|---|
| The locked schema from Step 5 | 5 of 5 | 2 of 5 |
| A refusal field, then the validator | 2 of 6 | 6 of 6 |

## Step 9: Test the extractor without calling the model

Each rule above gets a test that runs in milliseconds with no API key, so it can run on every
commit. If someone deletes `tool_choice` in a tidy up, loosens the schema, or drops a rule from the
validator, one of these tests fails.

![Test the extractor without calling the model](images/invoice-extractor-step-5.svg)

In [22]:
def test_request_forces_extract_invoice():
    request = build_extract_request("any document", REFUSING_TOOL)
    assert request["tool_choice"]["function"]["name"] == "extract_invoice"
    assert [tool["function"]["name"] for tool in request["tools"]] == ["extract_invoice"]


def test_schema_is_locked_down():
    schema = REFUSING_TOOL["function"]["parameters"]
    assert REFUSING_TOOL["function"]["strict"] is True
    assert schema["additionalProperties"] is False
    assert set(schema["required"]) == set(schema["properties"])
    for key in ("currency", "document_type"):
        assert "enum" in schema["properties"][key], f"{key} is free text again"


def test_type_check_rejects_a_total_written_as_text():
    reply = {**answers["northwind"], "total_cents": "334.26", "note": "paid by card"}
    assert len(find_schema_errors(reply, REFUSING_TOOL)) == 2


print("three tests defined for the request, the schema and the type check")

three tests defined for the request, the schema and the type check


The last three tests feed the checks replies of the kinds this notebook saw the model send, two of
them the recorded replies themselves, so each test guards against a failure that really happened
above.

In [23]:
def test_validator_rejects_francs_posted_as_euros():
    PAYABLES.clear()
    errors = find_answer_errors(locked["alpenlicht"], INVOICES["alpenlicht"])
    assert any("currency" in error for error in errors), errors


def test_validator_rejects_an_invoice_already_in_the_ledger():
    PAYABLES.clear()
    post_invoice(answers["northwind"])
    assert find_answer_errors(answers["northwind"], INVOICES["northwind"])


def test_conflicting_tool_calls_raise():
    calls = [SimpleNamespace(function=SimpleNamespace(arguments=json.dumps({"total_cents": n})))
             for n in (1100, 2200)]
    try:
        read_one_answer(SimpleNamespace(message=SimpleNamespace(tool_calls=calls)))
    except ConflictingAnswersError:
        return
    raise AssertionError("two different invoices in one reply were accepted")


print("three tests defined for the validator, the ledger and conflicting tool calls")

three tests defined for the validator, the ledger and conflicting tool calls


The next cell runs all six tests, and none of them calls the model.

In [24]:
for test in (test_request_forces_extract_invoice, test_schema_is_locked_down,
             test_type_check_rejects_a_total_written_as_text,
             test_validator_rejects_francs_posted_as_euros,
             test_validator_rejects_an_invoice_already_in_the_ledger,
             test_conflicting_tool_calls_raise):
    test()
    print(f"passed: {test.__name__}")

passed: test_request_forces_extract_invoice
passed: test_schema_is_locked_down
passed: test_type_check_rejects_a_total_written_as_text
passed: test_validator_rejects_francs_posted_as_euros
passed: test_validator_rejects_an_invoice_already_in_the_ledger
passed: test_conflicting_tool_calls_raise


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Forced tool_choice** | `build_extract_request` | Makes every reply arrive as an `extract_invoice` tool call, never as prose |
| **Syntactic validity** | `find_schema_errors` | Confirms the arguments parse, carry every required key and match the declared types |
| **Structural lock down** | `LOCKED_TOOL` | Integer cents, an `enum` of payable currencies, every key required, no extra keys, `strict` |
| **A legal refusal** | `document_type` in `REFUSING_TOOL` | Lets the model say a document is not an invoice or cannot be read |
| **One answer per reply** | `read_one_answer` | Accepts repeated identical tool calls and rejects calls that disagree |
| **Validator** | `find_answer_errors` | Rejects a valid answer whose values the document or the ledger contradicts |
| **Review queue** | `route_answer` and `NEEDS_REVIEW` | Posts only checked answers and sends everything else to a person with its reasons |